[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/multimodalidad/03-voz-ia.ipynb)

# Voz e IA: Transcripción y Síntesis de Voz

> Aprende a usar la API de OpenAI para transcribir audio con Whisper y sintetizar voz con TTS.
> Todo reproducible directamente en el notebook con `IPython.display.Audio`.

## Objetivos
- Transcribir audio desde URL pública con Whisper API
- Sintetizar voz con 3 voces diferentes usando el mismo texto
- Guardar archivos MP3 y escucharlos en el notebook
- Entender casos de uso de voz en aplicaciones IA

## 1. Instalación de dependencias

In [ ]:
%pip install openai requests --quiet

## 2. Configuración inicial

In [ ]:
import openai
import requests
import tempfile
from pathlib import Path
from IPython.display import Audio, display, HTML

client = openai.OpenAI()  # usa OPENAI_API_KEY

DIR_AUDIO = Path("audio_generado")
DIR_AUDIO.mkdir(exist_ok=True)

print("Cliente OpenAI inicializado.")
print(f"Archivos de audio se guardarán en: {DIR_AUDIO.absolute()}")
print("\nModelos disponibles:")
print("  Transcripción: whisper-1")
print("  Síntesis TTS: tts-1 (rápido), tts-1-hd (alta calidad)")

## 3. Transcripción de audio con Whisper

Whisper puede transcribir audio en más de 50 idiomas con alta precisión.
Descargamos un audio de muestra público para la demostración.

In [ ]:
def descargar_audio(url: str, nombre: str = "audio_descargado.mp3") -> Path:
    """Descarga un archivo de audio desde una URL."""
    ruta = DIR_AUDIO / nombre
    print(f"Descargando audio desde: {url}")
    respuesta = requests.get(url, timeout=30)
    respuesta.raise_for_status()
    ruta.write_bytes(respuesta.content)
    print(f"Audio guardado ({len(respuesta.content) / 1024:.0f} KB): {ruta}")
    return ruta


def transcribir_audio(ruta_audio: Path, idioma: str = None) -> dict:
    """Transcribe un archivo de audio con Whisper.

    Args:
        ruta_audio: Ruta al archivo de audio local
        idioma: Código ISO del idioma (ej: 'es', 'en'). None = autodetección.

    Returns:
        Dict con texto transcrito, idioma detectado y segmentos
    """
    print(f"Transcribiendo: {ruta_audio.name}")

    kwargs = {"model": "whisper-1", "response_format": "verbose_json"}
    if idioma:
        kwargs["language"] = idioma

    with open(ruta_audio, "rb") as f:
        transcripcion = client.audio.transcriptions.create(file=f, **kwargs)

    return {
        "texto": transcripcion.text,
        "idioma": transcripcion.language,
        "duracion_segundos": transcripcion.duration,
        "segmentos": len(transcripcion.segments) if transcripcion.segments else 0
    }


# Audio público de prueba (muestra de voz en español - Wikipedia Commons)
URL_AUDIO_PRUEBA = "https://upload.wikimedia.org/wikipedia/commons/5/5c/En-us-hello.ogg"

ruta_audio = descargar_audio(URL_AUDIO_PRUEBA, "muestra_hello.ogg")
print("\nReproducir audio original:")
display(Audio(str(ruta_audio)))

print("\nTranscribiendo...")
resultado_transcripcion = transcribir_audio(ruta_audio)

print("\n=== RESULTADO DE TRANSCRIPCIÓN ===")
print(f"Texto: '{resultado_transcripcion['texto']}'")
print(f"Idioma detectado: {resultado_transcripcion['idioma']}")
print(f"Duración: {resultado_transcripcion['duracion_segundos']:.1f} segundos")
print(f"Segmentos: {resultado_transcripcion['segmentos']}")

## 4. Síntesis de voz TTS con 3 voces diferentes

OpenAI ofrece 6 voces distintas. Comparamos 3 con el mismo texto en español.

In [ ]:
def sintetizar_voz(texto: str, voz: str, nombre_archivo: str,
                   modelo: str = "tts-1") -> Path:
    """Sintetiza texto a voz y guarda el MP3.

    Args:
        texto: Texto a sintetizar
        voz: Nombre de la voz (alloy, echo, fable, onyx, nova, shimmer)
        nombre_archivo: Nombre del archivo MP3 resultante
        modelo: 'tts-1' (rápido) o 'tts-1-hd' (alta calidad)

    Returns:
        Ruta al archivo MP3 generado
    """
    ruta = DIR_AUDIO / f"{nombre_archivo}.mp3"
    print(f"Sintetizando voz '{voz}'...")

    respuesta = client.audio.speech.create(
        model=modelo,
        voice=voz,
        input=texto
    )

    respuesta.stream_to_file(str(ruta))
    tamano_kb = ruta.stat().st_size / 1024
    print(f"  Guardado: {ruta.name} ({tamano_kb:.0f} KB)")
    return ruta


# Texto de demostración en español
TEXTO_DEMO = (
    "La inteligencia artificial está transformando el mundo. "
    "Hoy, las máquinas pueden escuchar, hablar y entender el lenguaje humano "
    "con una precisión sorprendente. Este es solo el comienzo de una revolución tecnológica."
)

# Las 6 voces disponibles y sus características
DESCRIPCION_VOCES = {
    "alloy":   "Neutra y versátil — buena para uso general",
    "echo":    "Masculina, clara y directa",
    "fable":   "Expresiva, cálida — ideal para narrativa",
    "onyx":    "Profunda y autoritaria",
    "nova":    "Femenina, amigable y optimista",
    "shimmer": "Femenina, suave y tranquila"
}

# Comparar 3 voces
VOCES_DEMO = ["alloy", "nova", "onyx"]

print("=" * 55)
print("SÍNTESIS DE VOZ: Comparando 3 voces")
print("=" * 55)
print(f"\nTexto: '{TEXTO_DEMO[:60]}...'")
print()

audios_generados = {}
for voz in VOCES_DEMO:
    ruta = sintetizar_voz(TEXTO_DEMO, voz, f"tts_{voz}")
    audios_generados[voz] = ruta

## 5. Reproducir y comparar las 3 voces en el notebook

In [ ]:
print("=== COMPARATIVA DE VOCES ===")
print(f"Texto sintetizado:\n'{TEXTO_DEMO}'\n")

for voz, ruta in audios_generados.items():
    descripcion = DESCRIPCION_VOCES[voz]
    tamano_kb = ruta.stat().st_size / 1024
    print(f"--- Voz: {voz.upper()} ---")
    print(f"Descripción: {descripcion}")
    print(f"Archivo: {ruta.name} ({tamano_kb:.0f} KB)")
    display(Audio(str(ruta)))
    print()

print("Todas las voces disponibles en OpenAI TTS:")
for voz, desc in DESCRIPCION_VOCES.items():
    marcador = "(*) " if voz in VOCES_DEMO else "    "
    print(f"  {marcador}{voz:10s}: {desc}")
print("\n(*) = voces demostradas en este notebook")

## 6. Comparativa TTS estándar vs alta calidad

In [ ]:
TEXTO_CORTO = "Bienvenido al futuro de la inteligencia artificial. La voz que escuchas es completamente sintética."

print("Comparando tts-1 (rápido) vs tts-1-hd (alta calidad) con la voz 'nova'...")

ruta_standard = sintetizar_voz(TEXTO_CORTO, "nova", "nova_standard", modelo="tts-1")
ruta_hd = sintetizar_voz(TEXTO_CORTO, "nova", "nova_hd", modelo="tts-1-hd")

print("\n--- tts-1 (estándar, más rápido y económico) ---")
display(Audio(str(ruta_standard)))

print("\n--- tts-1-hd (alta calidad, más natural) ---")
display(Audio(str(ruta_hd)))

print("\nConsejos de uso:")
print("  tts-1:    ideal para respuestas en tiempo real (chatbots, asistentes)")
print("  tts-1-hd: ideal para contenido pregrabado (narración, podcast, e-learning)")
print("\nArchivos generados en esta sesión:")
for archivo in sorted(DIR_AUDIO.glob("*.mp3")):
    print(f"  {archivo.name} ({archivo.stat().st_size / 1024:.0f} KB)")